# AdStream: Digital Advertising Allocation System

**A two-stage optimization system for digital ad inventory management**

## Core Data Model

**Bid Definition:** Each bid is a unique row identified by `bid_id`.
- Structure: `(bid_id, advertiser_id, topic, cost_per_impression, min_impressions, max_impressions)`
- An advertiser can submit multiple bids for the same topic with different parameters
- Each bid is evaluated independently (all-or-nothing: fully satisfied or rejected)

## Business Constraints

This system allocates advertising impressions to maximize revenue while respecting:
- **Bid minimums/maximums:** Each bid must receive ≥min impressions OR be rejected entirely
- **Topic matching:** Bids specify topics; placements must match bid topic to website topic
- **Diversification:** Each advertiser must appear on ≥MIN_WEBSITES_PER_ADVERTISER websites
- **Minimum placement:** ≥MIN_IMPRESSIONS_PER_PLACEMENT impressions per (advertiser, website) pair
- **Competitors:** Cannot be placed adjacent to each other (≥2 positions apart)
- **Violation penalty:** If ANY violations on a site, ALL revenue from that site gets 20% discount

## System Architecture

**Stage 1: Allocation (Combined Auction + Assignment)**
- Input: Advertiser bids (identified by bid_id)
- Output: (Advertiser, Website) placement decisions
- Objective: Maximize revenue
- Constraints: Website capacity, bid limits, ≥MIN_WEBSITES_PER_ADVERTISER websites per advertiser

**Stage 2: Ordering (Sequencing)**
- Input: Placement decisions from Stage 1
- Output: Advertiser sequence on each website
- Objective: Minimize competitor adjacency violations
- Constraints: ≥2 positions between competitors
- Penalty: If ANY violations on a site, ALL advertisers on that site get 20% discount

## Optimization Approaches

**Heuristic:** Advertiser-by-advertiser allocation (sorted by highest CPI) + random shuffle ordering

**Optimized:** Mixed-Integer Linear Programming (MILP) for allocation + Constraint Programming (CP) for ordering


## Setup

In [ ]:
# Install required packages
!pip install ortools pandas numpy plotly -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.7/27.7 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.1/321.1 kB 12.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.31.1 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.31.1 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.31.1 which is incompatible.


In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

NUM_WEBSITES = 100         # Number of websites with ad inventory
NUM_ADVERTISERS = 500      # Number of advertisers bidding
RANDOM_SEED = 42           # Random seed for reproducibility
VIOLATION_FINE_PERCENT = 0.20  # 20% discount on ALL revenue from sites with violations

# Diversification requirements
MIN_WEBSITES_PER_ADVERTISER = 10  # Each advertiser must appear on ≥MIN_WEBSITES_PER_ADVERTISER websites
MIN_IMPRESSIONS_PER_PLACEMENT = 1000  # Minimum impressions per (advertiser, website) placement
BLOCK_SIZE = 1000             # Block size for impression grouping in ordering

# Output Configuration
DEBUG_MODE = True            # Print detailed solver statistics and validation info
SHOW_PROGRESS = True         # Print progress messages during execution

# Note: Larger scale increases computational complexity

In [ ]:
from ortools.linear_solver import pywraplp
from ortools.sat.python import cp_model
import pandas as pd
import numpy as np
import random
import time
from datetime import datetime
from plotly.subplots import make_subplots
import plotly.graph_objects as go

random.seed(RANDOM_SEED)

# Formatting helpers
def format_number(n):
    """Format numbers with K suffix for thousands."""
    if n >= 1000:
        return f"{n/1000:.1f} K"
    return f"{n:.0f}"

def format_currency(n):
    """Format currency with K suffix for thousands."""
    if n >= 1000:
        return f"${n/1000:.1f} K"
    return f"${n:.2f}"

print(f"Setup complete: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Setup complete: 2025-11-20 03:49:04


## Data Generation

Generate synthetic ad market with:
- **Bids:** One bid per advertiser with unique `bid_id` (simplified structure)
- **Resource scarcity:** Limited capacity (10K-18K) vs bid minimums (8K-15K) creates competition
- **Flexible bounds:** Max = min * 2 (gives optimization room to work)
- **High CPI variance:** Wide range ($0.001-$0.100) creates clear value differences
- **Topic imbalance:** Sports/Tech are scarce (10% each), Finance/News are abundant (60%)
- **Demand pressure:** 60% of advertisers want scarce topics

Combined: Scarce capacity + high variance + one bid per advertiser = clear optimization opportunity.

In [ ]:
def generate_data(seed=RANDOM_SEED, num_websites=NUM_WEBSITES, num_advertisers=NUM_ADVERTISERS):
    """Generate synthetic ad allocation problem.

    Returns:
        dict with keys: websites, advertisers, bids, topics, competitor_pairs
    """
    random.seed(seed)
    np.random.seed(seed)

    TOPICS = ['Sports', 'Tech', 'Entertainment', 'Finance', 'News']

    # Create topic imbalance - some scarce, some abundant
    topic_distribution = {
        'Sports': 0.10,      # SCARCE - creates competition - creates competition (was 0.15)
        'Tech': 0.10,        # SCARCE - creates competition - creates competition (was 0.15)
        'Entertainment': 0.20,
        'Finance': 0.35,     # ABUNDANT - routing opportunity - routing opportunity (was 0.25)
        'News': 0.25         # ROUTING opportunity - routing opportunity (was 0.20)
    }

    # Generate websites
    website_data = []
    website_counter = 1
    for topic, proportion in topic_distribution.items():
        count = int(num_websites * proportion)
        for i in range(count):
            website_data.append({
                'website_id': f"{topic}_{website_counter}",
                'primary_topic': topic,
                'capacity': np.random.randint(10000, 18000),  # Reduced capacity creates scarcity
                'max_advertisers': np.random.randint(6, 13)
            })
            website_counter += 1

    websites_df = pd.DataFrame(website_data)

    # Generate advertisers with category focus
    category_distribution = {
        'Sports': 0.30,      # VERY HIGH demand for scarce topic
        'Tech': 0.30,        # VERY HIGH demand for scarce topic
        'Entertainment': 0.15,
        'Finance': 0.10,
        'General': 0.15      # Bids on multiple topics
    }

    advertiser_data = []
    adv_counter = 1
    for category, proportion in category_distribution.items():
        count = int(num_advertisers * proportion)
        for i in range(count):
            advertiser_data.append({
                'advertiser_id': f"Advertiser_{adv_counter}",
                'category': category
            })
            adv_counter += 1

    advertisers_df = pd.DataFrame(advertiser_data)

    # Generate bids - ONE bid per advertiser (simplified structure)
    bids_data = []
    bid_counter = 1
    for _, adv in advertisers_df.iterrows():
        adv_id = adv['advertiser_id']
        category = adv['category']

        # Choose topic based on category
        if category == 'General':
            # General advertisers bid on Finance or News (abundant topics)
            topic = np.random.choice(['Finance', 'News'])
            # Lower CPI for abundant topics
            cpi = np.random.uniform(0.001, 0.030)  # $0.001-$0.030 per impression
        else:
            # Specialized advertisers bid on their category
            topic = category
            if category in ['Sports', 'Tech']:
                # Premium pricing for scarce topics
                cpi = np.random.uniform(0.040, 0.100)  # $0.040-$0.100 per impression
            else:
                # Mid-range pricing for other topics
                cpi = np.random.uniform(0.015, 0.060)  # $0.015-$0.060 per impression

        # Set min impressions based on topic scarcity
        if topic in ['Sports', 'Tech']:
            min_impr = np.random.randint(8000, 12000)  # Smaller for scarce topics
        else:
            min_impr = np.random.randint(10000, 15000)  # Larger for abundant topics

        # Create single bid for this advertiser
        bids_data.append({
            'bid_id': f'Bid_{bid_counter}',
            'advertiser_id': adv_id,
            'topic': topic,
            'cost_per_impression': cpi,
            'min_impressions': min_impr,
            'max_impressions': min_impr * 2  # Flexible bounds: max = min * 2
        })
        bid_counter += 1

    bids_df = pd.DataFrame(bids_data)

    # Generate competitor pairs within categories
    competitor_pairs = []
    for category in category_distribution.keys():
        if category == 'General':
            continue
        category_advertisers = advertisers_df[advertisers_df['category'] == category]['advertiser_id'].tolist()
        for i in range(len(category_advertisers)):
            for j in range(i + 1, len(category_advertisers)):
                competitor_pairs.append((category_advertisers[i], category_advertisers[j]))

    print(f"Data Generation Complete:")
    print(f"  Websites:     {len(websites_df)}")
    print(f"  Advertisers:  {len(advertisers_df)}")
    print(f"  Bids:         {len(bids_df)}")
    print(f"  Competitors:  {len(competitor_pairs)} pairs")
    print(f"  Total Capacity: {format_number(websites_df['capacity'].sum())} impressions")

    return {
        'websites': websites_df,
        'advertisers': advertisers_df,
        'bids': bids_df,
        'topics': TOPICS,
        'competitor_pairs': competitor_pairs
    }

In [ ]:
# Generate data
data = generate_data()


Data Generation Complete:
  Websites:     100
  Advertisers:  500
  Bids:         500
  Competitors:  26350 pairs
  Total Capacity: 1423.0 K impressions


## System State Tracking

Class to track performance metrics across both stages.

In [ ]:
class SystemState:
    """Tracks system performance metrics."""

    def __init__(self):
        self.placements = {}
        self.sequences = {}
        self.violations = []
        self.gross_revenue = 0.0
        self.total_fines = 0.0
        self.net_revenue = 0.0
        self.violation_rate = 0.0
        self.allocation_time = 0.0
        self.ordering_time = 0.0

    def calculate_metrics(self, data):
        """Calculate all performance metrics with independent validation.

        Validation checks:
        1. Each bid meets its minimum impressions
        2. Each placement has ≥MIN_IMPRESSIONS_PER_PLACEMENT impressions
        3. Each advertiser appears on ≥MIN_WEBSITES_PER_ADVERTISER websites

        """
        # Step 1: Validate bid minimums
        bid_impressions = {}
        for (adv_id, website_id), impressions in self.placements.items():
            website_topic = data['websites'][data['websites']['website_id'] == website_id].iloc[0]['primary_topic']
            bid = data['bids'][(data['bids']['advertiser_id'] == adv_id) & (data['bids']['topic'] == website_topic)]
            if not bid.empty:
                bid_id = bid.iloc[0]['bid_id']
                bid_impressions[bid_id] = bid_impressions.get(bid_id, 0) + impressions

        valid_bids = set()
        for bid_id, total_impr in bid_impressions.items():
            bid = data['bids'][data['bids']['bid_id'] == bid_id].iloc[0]
            if total_impr >= int(bid['min_impressions']):
                valid_bids.add(bid_id)

        # Step 2: Build initial sets (only from valid bids with ≥MIN_IMPRESSIONS_PER_PLACEMENT)
        website_advertisers = {}  # website -> set of advertisers
        advertiser_websites = {}  # advertiser -> set of websites

        for (adv_id, website_id), impressions in self.placements.items():
            if impressions >= MIN_IMPRESSIONS_PER_PLACEMENT:
                website_topic = data['websites'][data['websites']['website_id'] == website_id].iloc[0]['primary_topic']
                bid = data['bids'][(data['bids']['advertiser_id'] == adv_id) & (data['bids']['topic'] == website_topic)]
                if not bid.empty and bid.iloc[0]["bid_id"] in valid_bids:
                    if website_id not in website_advertisers:
                        website_advertisers[website_id] = set()
                    website_advertisers[website_id].add(adv_id)

                    if adv_id not in advertiser_websites:
                        advertiser_websites[adv_id] = set()
                    advertiser_websites[adv_id].add(website_id)

        # Step 3: Check advertiser diversification
        # Websites: All websites with valid placements are valid
        valid_websites = set(website_advertisers.keys())

        # Advertisers: Only those appearing on ≥MIN_WEBSITES_PER_ADVERTISER websites are valid
        valid_advertisers = {a for a, webs in advertiser_websites.items()
                            if len(webs) >= MIN_WEBSITES_PER_ADVERTISER}

        if DEBUG_MODE:
            print(f"  [DEBUG] Validation: {len(valid_websites)} valid websites, {len(valid_advertisers)} valid advertisers")

        # Step 4: Calculate revenue (only from valid bids + valid websites + valid advertisers)
        self.gross_revenue = 0.0
        revenue_placements = 0

        for (adv_id, website_id), impressions in self.placements.items():
            if impressions < MIN_IMPRESSIONS_PER_PLACEMENT:
                continue
            if website_id not in valid_websites:
                continue
            if adv_id not in valid_advertisers:
                continue

            website_topic = data['websites'][data['websites']['website_id'] == website_id].iloc[0]['primary_topic']
            bid = data['bids'][(data['bids']['advertiser_id'] == adv_id) & (data['bids']['topic'] == website_topic)]
            if not bid.empty:
                bid_id = bid.iloc[0]['bid_id']
                if bid_id in valid_bids:
                    cost_per_impression = float(bid.iloc[0]['cost_per_impression'])
                    self.gross_revenue += impressions * cost_per_impression
                    revenue_placements += 1

        if DEBUG_MODE:
            print(f"  [DEBUG] Revenue: {revenue_placements} valid placements → ${self.gross_revenue/1000:.1f}K gross revenue")

        # Step 5: Calculate violations from sequences
        # Build competitor map
        competitor_map = {}
        for adv1, adv2 in data['competitor_pairs']:
            if adv1 not in competitor_map:
                competitor_map[adv1] = set()
            if adv2 not in competitor_map:
                competitor_map[adv2] = set()
            competitor_map[adv1].add(adv2)
            competitor_map[adv2].add(adv1)

        # Check for violations in sequences
        self.violations = []
        for website_id, blocks in self.sequences.items():
            for i in range(len(blocks) - 1):
                adv1, size1 = blocks[i]
                adv2, size2 = blocks[i + 1]
                if adv1 in competitor_map and adv2 in competitor_map[adv1]:
                    self.violations.append({
                        'website': website_id,
                        'pair': (adv1, adv2),
                        'block_sizes': (size1, size2)
                    })


        # Step 6: Calculate fines
        websites_with_violations = set(v["website"] for v in self.violations)
        self.violation_rate = len(websites_with_violations) / len(self.sequences) if self.sequences else 0

        self.total_fines = 0.0
        website_revenue = {}

        for (adv_id, website_id), impressions in self.placements.items():
            if impressions < MIN_IMPRESSIONS_PER_PLACEMENT:
                continue
            if website_id not in valid_websites:
                continue
            if adv_id not in valid_advertisers:
                continue

            website_topic = data['websites'][data['websites']['website_id'] == website_id].iloc[0]['primary_topic']
            bid = data['bids'][(data['bids']['advertiser_id'] == adv_id) & (data['bids']['topic'] == website_topic)]
            if not bid.empty:
                bid_id = bid.iloc[0]['bid_id']
                if bid_id in valid_bids:
                    cost_per_impression = float(bid.iloc[0]['cost_per_impression'])
                    revenue = impressions * cost_per_impression
                    website_revenue[website_id] = website_revenue.get(website_id, 0.0) + revenue

        for website_id in websites_with_violations:
            if website_id in website_revenue:
                self.total_fines += website_revenue[website_id] * VIOLATION_FINE_PERCENT

        self.net_revenue = self.gross_revenue - self.total_fines

---
# Stage 1: Allocation Component

Decide which advertisers are placed on which websites and how many impressions each gets.

## Heuristic Approach

**Simple greedy approach** - processes advertisers sequentially:

1. **Allocation:** Groups bids by advertiser, sorts advertisers by highest CPI, processes each advertiser fully before moving to next
2. **Ordering:** Random shuffle with retry

This represents a simpler system that handles advertisers one at a time, cannot globally optimize across all advertisers.

In [ ]:
def allocation_heuristic(data):
    """Greedy allocation: Process advertisers sequentially by value.

    Algorithm Strategy:
    - Priority queue: Sort advertisers by highest CPI (most valuable first)
    - Sequential processing: Fully process each advertiser before moving to next
    - Greedy capacity consumption: Each advertiser takes as much as possible (up to max_impr)
    - No look-ahead: Decisions are made for current advertiser without considering future advertisers

    Args:
        data: Dict containing:
            - bids: DataFrame with columns [bid_id, advertiser_id, topic, min_impressions,
                    max_impressions, cost_per_impression]
            - websites: DataFrame with columns [website_id, capacity, primary_topic]

    Returns:
        Dict[(advertiser_id, website_id)] -> int: Mapping of (advertiser, website) pairs to
            allocated impression counts. Only includes placements with impressions > 0.
    """
    # Group bids by advertiser
    advertiser_bids = {}
    for _, bid in data['bids'].iterrows():
        adv_id = bid['advertiser_id']
        if adv_id not in advertiser_bids:
            advertiser_bids[adv_id] = []
        advertiser_bids[adv_id].append(bid)

    # Sort advertisers by their highest CPI bid
    advertiser_max_cpi = {}
    for adv_id, bids in advertiser_bids.items():
        max_cpi = max(bid['cost_per_impression'] for bid in bids)
        advertiser_max_cpi[adv_id] = max_cpi

    sorted_advertisers = sorted(advertiser_max_cpi.items(), key=lambda x: x[1], reverse=True)

    website_capacity = {w['website_id']: int(w['capacity']) for _, w in data['websites'].iterrows()}
    final_placements = {}  # (advertiser, website) -> impressions

    # Process each advertiser in order
    for adv_id, max_cpi in sorted_advertisers:
        bids = advertiser_bids[adv_id]

        # Sort this advertiser's bids by CPI (highest first)
        sorted_bids = sorted(bids, key=lambda b: b['cost_per_impression'], reverse=True)

        # Try to place each bid for this advertiser
        for bid in sorted_bids:
            topic = bid['topic']
            min_impr = int(bid['min_impressions'])
            max_impr = int(bid['max_impressions'])

            # Find eligible websites for this topic
            eligible = data['websites'][data['websites']['primary_topic'] == topic]['website_id'].tolist()
            if not eligible:
                continue

            # Try to allocate this bid
            total_allocated = 0
            site_idx = 0

            # Allocate round-robin in blocks of MIN_IMPRESSIONS_PER_PLACEMENT
            while total_allocated < max_impr and site_idx < len(eligible) * 1000:
                website_id = eligible[site_idx % len(eligible)]
                site_capacity = website_capacity.get(website_id, 0)

                if site_capacity >= MIN_IMPRESSIONS_PER_PLACEMENT:
                    remaining = max_impr - total_allocated
                    if remaining < MIN_IMPRESSIONS_PER_PLACEMENT:
                        break
                    to_place = min(MIN_IMPRESSIONS_PER_PLACEMENT, remaining, site_capacity)
                    final_placements[(adv_id, website_id)] = final_placements.get((adv_id, website_id), 0) + to_place
                    total_allocated += to_place
                    website_capacity[website_id] -= to_place

                site_idx += 1

            # Accept bid if minimum met (don't need to track, just place)

    return final_placements


## Optimized Approach

In [ ]:
from ortools.linear_solver import pywraplp

def allocation_optimized(data):
    """
    Stage 1: global allocation optimization.

    Maximizes total revenue subject to:
      - Website capacity limits
      - Optional advertisers with min/max total impressions
      - Min websites per *served* advertiser
      - Min impressions per placement
      - Topic match between advertiser bids and website primary_topic

    Advertisers are optional: if capacity is tight, the model can give an
    advertiser 0 impressions instead of partially serving them.
    """
    bids = data["bids"].copy()
    websites = data["websites"].copy()

    advertisers = sorted(bids["advertiser_id"].unique())
    sites = sorted(websites["website_id"].unique())

    # Website attributes
    site_topic = websites.set_index("website_id")["primary_topic"].to_dict()
    site_cap   = websites.set_index("website_id")["capacity"].astype(int).to_dict()

    # Aggregate min / max impressions per advertiser
    adv_min = (
        bids.groupby("advertiser_id")["min_impressions"]
        .sum()
        .astype(int)
        .to_dict()
    )
    adv_max = (
        bids.groupby("advertiser_id")["max_impressions"]
        .sum()
        .astype(int)
        .to_dict()
    )

    # Best CPI per (advertiser, topic)
    best_cpi_topic = (
        bids.groupby(["advertiser_id", "topic"])["cost_per_impression"]
        .max()
        .to_dict()
    )

    # Build allowed (advertiser, website) pairs: topic must match
    pairs = []
    cpi = {}
    for (a, t), best_cpi in best_cpi_topic.items():
        for w in sites:
            if site_topic[w] == t:
                pairs.append((a, w))
                cpi[(a, w)] = float(best_cpi)

    if not pairs:
        return {}

    # Create solver
    solver = pywraplp.Solver.CreateSolver("CBC")
    if solver is None:
        return None

    # Decision variables
    x = {}  # impressions allocated to (a,w)
    y = {}  # 1 if advertiser a uses website w
    z = {}  # 1 if advertiser a is active (served at all)

    adv_sites = {a: [] for a in advertisers}
    site_advs = {w: [] for w in sites}

    # Create x and y with tight upper bounds
    for a, w in pairs:
        max_req = int(adv_max.get(a, 0))
        if max_req <= 0:
            continue
        ub = int(min(site_cap[w], max_req))
        if ub <= 0:
            continue

        x[(a, w)] = solver.IntVar(0, ub, f"x_{a}_{w}")
        y[(a, w)] = solver.BoolVar(f"y_{a}_{w}")
        adv_sites[a].append(w)
        site_advs[w].append(a)

    # Advertiser activity vars
    for a in advertisers:
        z[a] = solver.BoolVar(f"z_{a}")

    # If no x vars, nothing to allocate
    if not x:
        return {}

    # Website capacity constraints
    for w in sites:
        if site_advs[w]:
            solver.Add(
                sum(x[(a, w)] for a in site_advs[w]) <= site_cap[w]
            )

    # Link x and y, enforce min placement size on each (a,w)
    for (a, w), x_var in x.items():
        y_var = y[(a, w)]
        ub = int(min(site_cap[w], adv_max.get(a, 0)))
        solver.Add(x_var >= MIN_IMPRESSIONS_PER_PLACEMENT * y_var)
        solver.Add(x_var <= ub * y_var)

    # Advertiser-level constraints (optional advertisers)
    for a in advertisers:
        if not adv_sites[a]:
            # No eligible sites → advertiser must be inactive
            solver.Add(z[a] == 0)
            continue

        total_impr = sum(x[(a, w)] for w in adv_sites[a])
        min_req = int(adv_min.get(a, 0))
        max_req = int(adv_max.get(a, 0))

        # If active, must hit their min
        if min_req > 0:
            solver.Add(total_impr >= min_req * z[a])

        # If inactive (z=0), forces total_impr = 0; if active, upper-bounded by max_req
        if max_req > 0:
            solver.Add(total_impr <= max_req * z[a])
        else:
            # No explicit max: just tie activity to having impressions
            solver.Add(
                total_impr <= sum(site_cap[w] for w in adv_sites[a]) * z[a]
            )

        # Min websites per active advertiser
        if MIN_WEBSITES_PER_ADVERTISER > 0:
            solver.Add(
                sum(y[(a, w)] for w in adv_sites[a])
                >= MIN_WEBSITES_PER_ADVERTISER * z[a]
            )

    # Objective: maximize total revenue
    solver.Maximize(
        sum(cpi[(a, w)] * x[(a, w)] for (a, w) in x.keys())
    )

    status = solver.Solve()
    if status not in (pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE):
        return None

    # Extract solution
    final_placements = {}
    for (a, w), x_var in x.items():
        val = int(round(x_var.solution_value()))
        if val > 0:
            final_placements[(a, w)] = val

    return final_placements


## Allocation Check

In [ ]:
def evaluate_allocation(data, allocation, name="solution"):
    """
    Check feasibility and quality of a given allocation.
    """
    print(f"\n=== Evaluation: {name} ===")
    if allocation is None:
        print("❌ No solution (allocation is None).")
        return {"feasible": False}

    bids = data["bids"]
    websites = data["websites"]

    site_topic = websites.set_index("website_id")["primary_topic"].to_dict()
    site_cap   = websites.set_index("website_id")["capacity"].astype(int).to_dict()

    adv_min = bids.groupby("advertiser_id")["min_impressions"].sum().astype(int).to_dict()
    adv_max = bids.groupby("advertiser_id")["max_impressions"].sum().astype(int).to_dict()

    best_cpi_topic = (
        bids.groupby(["advertiser_id", "topic"])["cost_per_impression"]
        .max()
        .to_dict()
    )

    website_load = {w: 0 for w in site_cap.keys()}
    adv_total_impr = {}
    adv_sites = {}

    total_revenue = 0.0
    total_impr = 0

    min_placement_violations = 0
    topic_mismatch_violations = 0

    for (a, w), impr in allocation.items():
        impr = int(impr)
        if impr <= 0:
            continue

        total_impr += impr
        website_load[w] = website_load.get(w, 0) + impr
        adv_total_impr[a] = adv_total_impr.get(a, 0) + impr
        adv_sites.setdefault(a, set()).add(w)

        if impr < MIN_IMPRESSIONS_PER_PLACEMENT:
            min_placement_violations += 1

        topic = site_topic[w]
        if (a, topic) not in best_cpi_topic:
            topic_mismatch_violations += 1

        cpi = best_cpi_topic.get((a, topic), 0.0)
        total_revenue += cpi * impr

    # Capacity violations
    capacity_violations = 0
    for w, load in website_load.items():
        if load > site_cap[w]:
            capacity_violations += 1

    # Advertiser min/max + min websites
    adv_min_viol = 0
    adv_max_viol = 0
    adv_site_viol = 0
    unserved = 0

    advertisers = bids["advertiser_id"].unique()
    for a in advertisers:
        total = adv_total_impr.get(a, 0)
        min_req = int(adv_min.get(a, 0))
        max_req = int(adv_max.get(a, 0))
        site_count = len(adv_sites.get(a, set()))

        if min_req > 0:
            if total == 0:
                unserved += 1
            elif total < min_req:
                adv_min_viol += 1

        if max_req > 0 and total > max_req:
            adv_max_viol += 1

        if total > 0 and site_count < MIN_WEBSITES_PER_ADVERTISER:
            adv_site_viol += 1

    feasible = (
        capacity_violations == 0
        and adv_min_viol == 0
        and adv_max_viol == 0
        and adv_site_viol == 0
        and min_placement_violations == 0
        and topic_mismatch_violations == 0
    )

    print(f"Total impressions                   : {total_impr}")
    print(f"Total revenue                       : {total_revenue:,.2f}")
    print(f"Feasible                            : {feasible}")
    print(f"- Capacity violations               : {capacity_violations}")
    print(f"- Advertiser MIN violations         : {adv_min_viol}")
    print(f"- Advertiser MAX violations         : {adv_max_viol}")
    print(f"- Min-websites violations           : {adv_site_viol}")
    print(f"- Min-placement violations          : {min_placement_violations}")
    print(f"- Topic-mismatch violations         : {topic_mismatch_violations}")
    print(f"- Unserved advertisers (min>0, 0impr): {unserved}")

    return {
        "name": name,
        "feasible": feasible,
        "total_impressions": total_impr,
        "total_revenue": total_revenue,
        "capacity_violations": capacity_violations,
        "adv_min_violations": adv_min_viol,
        "adv_max_violations": adv_max_viol,
        "adv_site_violations": adv_site_viol,
        "min_placement_violations": min_placement_violations,
        "topic_mismatch_violations": topic_mismatch_violations,
        "unserved_advertisers": unserved,
    }


def compare_allocations(data):
    """
    Run both heuristic and optimized allocation and compare them.
    """
    heur_alloc = allocation_heuristic(data)
    opt_alloc = allocation_optimized(data)

    heur_metrics = evaluate_allocation(data, heur_alloc, name="Heuristic")
    opt_metrics = evaluate_allocation(data, opt_alloc, name="Optimized")

    if heur_metrics.get("feasible") and opt_metrics.get("feasible"):
        rev_h = heur_metrics["total_revenue"]
        rev_o = opt_metrics["total_revenue"]
        print("\n=== Allocation Revenue Comparison ===")
        print(f"Heuristic revenue : {rev_h:,.2f}")
        print(f"Optimized revenue : {rev_o:,.2f}")
        if rev_h > 0:
            lift = (rev_o - rev_h) / rev_h * 100
            print(f"Relative lift     : {lift:.2f}%")
    return heur_metrics, opt_metrics


In [ ]:
heur_metrics, opt_metrics = compare_allocations(data)



=== Evaluation: Heuristic ===
Total impressions                   : 1374000
Total revenue                       : 77,097.21
Feasible                            : False
- Capacity violations               : 0
- Advertiser MIN violations         : 2
- Advertiser MAX violations         : 0
- Min-websites violations           : 8
- Min-placement violations          : 0
- Topic-mismatch violations         : 0
- Unserved advertisers (min>0, 0impr): 439

=== Evaluation: Optimized ===
Total impressions                   : 1422998
Total revenue                       : 79,672.34
Feasible                            : True
- Capacity violations               : 0
- Advertiser MIN violations         : 0
- Advertiser MAX violations         : 0
- Min-websites violations           : 0
- Min-placement violations          : 0
- Topic-mismatch violations         : 0
- Unserved advertisers (min>0, 0impr): 438


---
# Stage 2: Ordering Component

Sequence advertisers on each website to minimize competitor adjacency.

**Approach:** Group impressions into 1000-impression blocks, then sequence blocks

**Constraint:** Competitor blocks must be ≥2 positions apart

**Objective:** Achieve zero violations per website

**Penalty:** If a website has ANY violations, ALL revenue from that website is discounted by 20%

## Heuristic Approach

Random shuffle of 1000-impression blocks.

In [ ]:
def ordering_heuristic(data, placements):
    """Random shuffle ordering: Simple block-based sequencing.

    Algorithm Strategy:
    - Block creation: Convert impressions to 1000-impression blocks
    - Random ordering: Shuffle blocks randomly on each website

    Args:
        data: Dict containing:
            - competitor_pairs: List of tuples (advertiser_id_1, advertiser_id_2) indicating
                                competing advertisers that should not be adjacent
        placements: Dict[(advertiser_id, website_id)] -> int: Impression allocations from
                    allocation stage. Maps (advertiser, website) pairs to impression counts.

    Returns:
        Dict[website_id] -> List[(advertiser_id, block_size)]: Mapping of website IDs to
            ordered sequences of advertisement blocks. Each block is a tuple of
            (advertiser_id, impression_count).
    """

    # Group by website
    website_advertisers = {}
    for (adv_id, website_id), impressions in placements.items():
        if website_id not in website_advertisers:
            website_advertisers[website_id] = []
        website_advertisers[website_id].append((adv_id, impressions))

    # Build competitor map
    competitor_map = {}
    for adv1, adv2 in data['competitor_pairs']:
        if adv1 not in competitor_map:
            competitor_map[adv1] = set()
        if adv2 not in competitor_map:
            competitor_map[adv2] = set()
        competitor_map[adv1].add(adv2)
        competitor_map[adv2].add(adv1)

    sequences = {}

    for website_id, adv_impressions in website_advertisers.items():
        # Create blocks of impressions
        blocks = []
        for adv_id, total_impressions in adv_impressions:
            remaining = total_impressions
            while remaining > 0:
                block_size = min(BLOCK_SIZE, remaining)
                blocks.append((adv_id, block_size))
                remaining -= block_size

        # Random shuffle
        random.shuffle(blocks)
        sequences[website_id] = blocks

    return sequences

## Optimized Approach

In [ ]:
def ordering_optimized(data, placements):
    """
    Improved Stage 2: local-search ordering.

    For each website:
      1. Build 1000-impression blocks (same as heuristic).
      2. Start from a random ordering of blocks.
      3. Run local search that tries to move blocks around to reduce
         competitor adjacencies, never making them worse.

    This does NOT guarantee zero violations (some websites are inherently
    impossible), but it tends to strictly reduce violations compared to
    a pure random ordering.
    """
    # --- Group placements by website ---
    website_advertisers = {}
    for (adv_id, website_id), impressions in placements.items():
        if website_id not in website_advertisers:
            website_advertisers[website_id] = []
        website_advertisers[website_id].append((adv_id, impressions))

    # --- Build competitor map ---
    competitor_map = {}
    for adv1, adv2 in data.get("competitor_pairs", []):
        competitor_map.setdefault(adv1, set()).add(adv2)
        competitor_map.setdefault(adv2, set()).add(adv1)

    def are_competitors(a, b):
        return b in competitor_map.get(a, set())

    # --- Helpers: count & improve sequence ---
    def count_violations(blocks):
        v = 0
        for i in range(len(blocks) - 1):
            if are_competitors(blocks[i][0], blocks[i + 1][0]):
                v += 1
        return v

    def improve_sequence(blocks, max_iters=2000):
        """
        Simple local search: repeatedly move blocks involved in violations
        to positions that reduce the total number of violations.

        Returns a sequence that has <= violations than the input.
        """
        best = list(blocks)
        best_v = count_violations(best)
        if best_v == 0:
            return best

        n = len(best)

        for _ in range(max_iters):
            improved = False
            # Scan for a violating adjacency
            for i in range(n - 1):
                a1 = best[i][0]
                a2 = best[i + 1][0]
                if not are_competitors(a1, a2):
                    continue  # no violation here

                # Try moving the second block (i+1) somewhere else
                for j in range(n):
                    if j == i or j == i + 1:
                        continue
                    new_seq = best.copy()
                    block = new_seq.pop(i + 1)
                    # Adjust target index after pop
                    if j > i + 1:
                        j -= 1
                    new_seq.insert(j, block)

                    new_v = count_violations(new_seq)
                    if new_v < best_v:
                        best, best_v = new_seq, new_v
                        n = len(best)
                        improved = True
                        break

                if improved:
                    break

            if not improved or best_v == 0:
                break

        return best

    # --- Build sequences per website ---
    sequences = {}

    for website_id, adv_impressions in website_advertisers.items():
        # Create blocks
        blocks = []
        for adv_id, total_impressions in adv_impressions:
            remaining = total_impressions
            while remaining > 0:
                block_size = min(BLOCK_SIZE, remaining)
                blocks.append((adv_id, block_size))
                remaining -= block_size

        if not blocks:
            sequences[website_id] = []
            continue

        # Initial random ordering (same idea as heuristic)
        random.shuffle(blocks)

        # Local search to clean up competitor adjacencies
        improved_blocks = improve_sequence(blocks)

        sequences[website_id] = improved_blocks

    return sequences



## Ordering Check

In [ ]:
from collections import defaultdict

REVENUE_PENALTY_FACTOR = 0.8  # 20% discount if any violation on a website

def evaluate_ordering(data, placements, sequences, name="Ordering"):
    """
    Check an ordering solution for:
      - Impression conservation (blocks sum to placements)
      - Competitor adjacency violations
      - Revenue before and after 20% brand-safety penalty

    Args:
        data: dict with at least 'bids', 'websites', 'competitor_pairs'
        placements: Dict[(adv_id, website_id)] -> impressions (from allocation)
        sequences: Dict[website_id] -> List[(adv_id, block_size)]
        name: label for printing

    Returns:
        metrics: dict with violation counts and revenue numbers.
    """
    print(f"\n=== Ordering Evaluation: {name} ===")

    bids = data["bids"]
    websites = data["websites"]

    # --- Helper maps ---
    site_topic = websites.set_index("website_id")["primary_topic"].to_dict()

    # Best CPI per (advertiser, topic) for revenue
    best_cpi_topic = (
        bids.groupby(["advertiser_id", "topic"])["cost_per_impression"]
        .max()
        .to_dict()
    )

    # Competitor map
    competitor_map = {}
    for a, b in data.get("competitor_pairs", []):
        competitor_map.setdefault(a, set()).add(b)
        competitor_map.setdefault(b, set()).add(a)

    def are_competitors(a, b):
        return b in competitor_map.get(a, set())

    # --- 1) Check impression conservation ---
    # Sum blocks back into (adv, website) totals
    placed_impressions = defaultdict(int)
    for w, blocks in sequences.items():
        for adv, block_size in blocks:
            placed_impressions[(adv, w)] += int(block_size)

    # Compare with placements
    mismatch_pairs = 0
    mismatch_amount = 0

    all_keys = set(placements.keys()) | set(placed_impressions.keys())
    for key in all_keys:
        target = int(placements.get(key, 0))
        actual = int(placed_impressions.get(key, 0))
        if target != actual:
            mismatch_pairs += 1
            mismatch_amount += abs(target - actual)

    # --- 2) Competitor adjacency violations per website ---
    total_violations = 0
    websites_with_violation = set()

    for w, blocks in sequences.items():
        for i in range(len(blocks) - 1):
            adv1, _ = blocks[i]
            adv2, _ = blocks[i + 1]
            if are_competitors(adv1, adv2):
                total_violations += 1
                websites_with_violation.add(w)

    # --- 3) Revenue before and after penalties ---
    base_revenue_per_site = defaultdict(float)
    total_base_revenue = 0.0

    for (adv, w), impr in placements.items():
        topic = site_topic[w]
        cpi = best_cpi_topic.get((adv, topic), 0.0)
        rev = cpi * int(impr)
        base_revenue_per_site[w] += rev
        total_base_revenue += rev

    penalized_revenue_per_site = {}
    total_penalized_revenue = 0.0

    for w, rev in base_revenue_per_site.items():
        if w in websites_with_violation:
            rev_after = rev * REVENUE_PENALTY_FACTOR
        else:
            rev_after = rev
        penalized_revenue_per_site[w] = rev_after
        total_penalized_revenue += rev_after

    # --- Print summary ---
    print(f"Impression mismatch pairs           : {mismatch_pairs}")
    print(f"Total mismatch impressions          : {mismatch_amount}")
    print(f"Competitor adjacency violations     : {total_violations}")
    print(f"Websites with ≥1 violation          : {len(websites_with_violation)}")
    print(f"Base revenue (no penalties)         : {total_base_revenue:,.2f}")
    print(f"Revenue after 20% penalties         : {total_penalized_revenue:,.2f}")

    return {
        "name": name,
        "mismatch_pairs": mismatch_pairs,
        "mismatch_impressions": mismatch_amount,
        "competitor_violations": total_violations,
        "websites_with_violation": len(websites_with_violation),
        "base_revenue": total_base_revenue,
        "penalized_revenue": total_penalized_revenue,
    }


def compare_orderings(data, placements):
    """
    Run both heuristic and optimized ordering and compare them.
    Assumes allocation has already been done.

    Args:
        data: problem data dict
        placements: output of allocation_heuristic or allocation_optimized

    Returns:
        (heur_metrics, opt_metrics)
    """
    seq_heur = ordering_heuristic(data, placements)
    seq_opt = ordering_optimized(data, placements)

    heur_metrics = evaluate_ordering(data, placements, seq_heur, name="Heuristic ordering")
    opt_metrics = evaluate_ordering(data, placements, seq_opt, name="Optimized ordering")

    # Quick comparison
    print("\n=== Ordering Comparison Summary ===")
    print(f"Heuristic penalized revenue : {heur_metrics['penalized_revenue']:,.2f}")
    print(f"Optimized penalized revenue : {opt_metrics['penalized_revenue']:,.2f}")
    print(f"Heuristic violations        : {heur_metrics['competitor_violations']}")
    print(f"Optimized violations        : {opt_metrics['competitor_violations']}")

    return heur_metrics, opt_metrics


In [ ]:
placements_opt = allocation_optimized(data)
heur_metrics, opt_metrics = compare_orderings(data, placements_opt)



=== Ordering Evaluation: Heuristic ordering ===
Impression mismatch pairs           : 0
Total mismatch impressions          : 0
Competitor adjacency violations     : 680
Websites with ≥1 violation          : 73
Base revenue (no penalties)         : 79,672.34
Revenue after 20% penalties         : 65,936.70

=== Ordering Evaluation: Optimized ordering ===
Impression mismatch pairs           : 0
Total mismatch impressions          : 0
Competitor adjacency violations     : 466
Websites with ≥1 violation          : 73
Base revenue (no penalties)         : 79,672.34
Revenue after 20% penalties         : 65,936.70

=== Ordering Comparison Summary ===
Heuristic penalized revenue : 65,936.70
Optimized penalized revenue : 65,936.70
Heuristic violations        : 680
Optimized violations        : 466


---
# Run System

Execute the two-stage pipeline and calculate performance metrics.

In [ ]:
def run_system(data, use_optimization=False):
    """Run the two-stage system."""
    state = SystemState()

    print("\n" + "="*60)
    print(f"Running System: {'OPTIMIZED' if use_optimization else 'HEURISTIC'}")
    print("="*60)

    # Stage 1: Allocation
    if SHOW_PROGRESS:
        print("\n[1/2] Allocation Component...")
    start_time = time.time()
    if use_optimization:
        state.placements = allocation_optimized(data)
        if state.placements is None:
            print("\n" + "="*60)
            print("="*60)
            print("="*60)
            return None
    else:
        state.placements = allocation_heuristic(data)
    state.allocation_time = time.time() - start_time
    if SHOW_PROGRESS:
        print(f"  ✓ Placed {format_number(sum(state.placements.values()))} impressions across {len(state.placements)} (advertiser, website) pairs")
        print(f"  ✓ Allocation time: {state.allocation_time:.2f}s")

    # Stage 2: Ordering
    if SHOW_PROGRESS:
        print("\n[2/2] Ordering Component...")
    start_time = time.time()
    if use_optimization:
        state.sequences = ordering_optimized(data, state.placements)
        if state.sequences is None:
            print("\n" + "="*60)
            print("Stage 2 (Ordering) returned no solution.")
            print("Check ordering_optimized() implementation.")
            print("="*60)
            return None
    else:
        state.sequences = ordering_heuristic(data, state.placements)
    state.ordering_time = time.time() - start_time
    if SHOW_PROGRESS:
        print(f"  ✓ Sequenced advertisers on {len(state.sequences)} websites")
        print(f"  ✓ Ordering time: {state.ordering_time:.2f}s")

    # Calculate metrics
    state.calculate_metrics(data)

    # Print summary
    total_capacity = data['websites']['capacity'].sum()
    placed_impressions = sum(state.placements.values())

    # Advertisers meeting requirements is calculated in calculate_metrics

    print("\n" + "="*60)
    print("System Performance Summary:")
    print("="*60)
    print(f"\nAllocation Component:")
    print(f"  Placed Impressions:      {format_number(placed_impressions)} / {format_number(total_capacity)}")
    print(f"  Utilization:             {placed_impressions/total_capacity:.1%}")
    print(f"  Gross Revenue:           {format_currency(state.gross_revenue)}")
    print(f"  Time:                    {state.allocation_time:.2f}s")
    print(f"\nOrdering Component:")
    websites_with_violations = len(set(v['website'] for v in state.violations))
    print(f"  Sites with violations:   {websites_with_violations} / {len(state.sequences)}")
    print(f"  Violation Rate:          {state.violation_rate:.1%}")
    print(f"  Violation Fines:         {format_currency(state.total_fines)} ({VIOLATION_FINE_PERCENT:.0%} discount on all violating sites)")
    print(f"  Time:                    {state.ordering_time:.2f}s")
    print(f"\n" + "-"*60)
    print(f"NET REVENUE:               {format_currency(state.net_revenue)}")
    print(f"TOTAL TIME:                {state.allocation_time + state.ordering_time:.2f}s")
    print("="*60)

    return state

## Execute System

Run both heuristic and optimized versions to compare performance.

In [ ]:
# Run heuristic
heuristic_state = run_system(data, use_optimization=False)


Running System: HEURISTIC

[1/2] Allocation Component...
  ✓ Placed 1374.0 K impressions across 1064 (advertiser, website) pairs
  ✓ Allocation time: 1.66s

[2/2] Ordering Component...
  ✓ Sequenced advertisers on 100 websites
  ✓ Ordering time: 0.01s
  [DEBUG] Validation: 100 valid websites, 53 valid advertisers
  [DEBUG] Revenue: 1011 valid placements → $67.3K gross revenue

System Performance Summary:

Allocation Component:
  Placed Impressions:      1374.0 K / 1423.0 K
  Utilization:             96.6%
  Gross Revenue:           $67.3 K
  Time:                    1.66s

Ordering Component:
  Sites with violations:   75 / 100
  Violation Rate:          75.0%
  Violation Fines:         $11.7 K (20% discount on all violating sites)
  Time:                    0.01s

------------------------------------------------------------
NET REVENUE:               $55.7 K
TOTAL TIME:                1.67s


In [ ]:
# Run optimized
optimized_state = run_system(data, use_optimization=True)


Running System: OPTIMIZED

[1/2] Allocation Component...
  ✓ Placed 1423.0 K impressions across 637 (advertiser, website) pairs
  ✓ Allocation time: 6.97s

[2/2] Ordering Component...
  ✓ Sequenced advertisers on 100 websites
  ✓ Ordering time: 0.06s
  [DEBUG] Validation: 100 valid websites, 62 valid advertisers
  [DEBUG] Revenue: 637 valid placements → $79.7K gross revenue

System Performance Summary:

Allocation Component:
  Placed Impressions:      1423.0 K / 1423.0 K
  Utilization:             100.0%
  Gross Revenue:           $79.7 K
  Time:                    6.97s

Ordering Component:
  Sites with violations:   73 / 100
  Violation Rate:          73.0%
  Violation Fines:         $13.7 K (20% discount on all violating sites)
  Time:                    0.06s

------------------------------------------------------------
NET REVENUE:               $65.9 K
TOTAL TIME:                7.03s


## Performance Comparison

In [ ]:
print("\n" + "="*60)
print("HEURISTIC vs OPTIMIZED COMPARISON")
print("="*60)

# Check if both states are valid
if heuristic_state is None:
    print("\n[ERROR] Heuristic state is None - cannot perform comparison")
elif optimized_state is None:
    print("\n[ERROR] Optimized state is None - cannot perform comparison")
    print("The optimized approach failed. Only heuristic results are available.")
else:
    print(f"\nGross Revenue:")
    print(f"  Heuristic:  {format_currency(heuristic_state.gross_revenue)}")
    print(f"  Optimized:  {format_currency(optimized_state.gross_revenue)}")
    print(f"  Difference: {format_currency(optimized_state.gross_revenue - heuristic_state.gross_revenue)}")

    print(f"\nViolation Fines:")
    print(f"  Heuristic:  {format_currency(heuristic_state.total_fines)}")
    print(f"  Optimized:  {format_currency(optimized_state.total_fines)}")
    print(f"  Savings:    {format_currency(heuristic_state.total_fines - optimized_state.total_fines)}")

    print(f"\nNet Revenue:")
    print(f"  Heuristic:  {format_currency(heuristic_state.net_revenue)}")
    print(f"  Optimized:  {format_currency(optimized_state.net_revenue)}")
    improvement = optimized_state.net_revenue - heuristic_state.net_revenue
    pct = ((optimized_state.net_revenue / heuristic_state.net_revenue - 1) * 100) if heuristic_state.net_revenue != 0 else 0
    print(f"  Improvement: {format_currency(improvement)} ({pct:+.1f}%)")


    print(f"\nViolation Rate:")
    print(f"  Heuristic:  {heuristic_state.violation_rate:.1%}")
    print(f"  Optimized:  {optimized_state.violation_rate:.1%}")

    print(f"\nTotal Time:")
    heuristic_total = heuristic_state.allocation_time + heuristic_state.ordering_time
    optimized_total = optimized_state.allocation_time + optimized_state.ordering_time
    print(f"  Heuristic:  {heuristic_total:.2f}s")
    print(f"  Optimized:  {optimized_total:.2f}s")
    print(f"  Difference: {optimized_total - heuristic_total:+.2f}s")
print("="*60)


HEURISTIC vs OPTIMIZED COMPARISON

Gross Revenue:
  Heuristic:  $67.3 K
  Optimized:  $79.7 K
  Difference: $12.3 K

Violation Fines:
  Heuristic:  $11.7 K
  Optimized:  $13.7 K
  Savings:    $-2053.45

Net Revenue:
  Heuristic:  $55.7 K
  Optimized:  $65.9 K
  Improvement: $10.3 K (+18.5%)

Violation Rate:
  Heuristic:  75.0%
  Optimized:  73.0%

Total Time:
  Heuristic:  1.67s
  Optimized:  7.03s
  Difference: +5.36s
